# 03 — Collision Risk Model (ESA Kelvins Dataset)

Notebook 02 gave us a fast pipeline that finds conjunction events.
But a miss distance of 5 km means very different things depending on closing velocity,
object sizes, and how stale the orbit determination is.

This notebook trains a **gradient-boosted regressor** on **real conjunction data** from
the [ESA Kelvins Collision Avoidance Challenge](https://kelvins.esa.int/collision-avoidance-challenge/).

### Dataset
- **Source:** ESA Space Debris Office, archived on [Zenodo](https://zenodo.org/records/4463683)
- **Contents:** ~162k real Conjunction Data Messages (CDMs) across ~13k events
- **Target:** `risk` — log₁₀ collision probability from the final CDM before closest approach
- **Features:** 103 columns including miss distance, relative velocity, covariance matrices,
  object properties, orbit determination quality, and space weather indices

### Approach
Each conjunction event has a series of CDMs issued at different times before TCA.
For the real-time pipeline, we care about: **given the latest CDM snapshot, what is the risk?**
So we take the most recent CDM per event and train on that.

In [ ]:
!pip install scikit-learn scipy requests zipfile-deflate64

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import requests
import zipfile
import os

np.random.seed(42)
print('All imports successful.')

---
## 1. Download the ESA Kelvins Dataset

The dataset is archived on Zenodo under CC-BY 4.0.
We download and extract it into `data/kelvins/`.

In [ ]:
DATA_DIR = Path('..') / 'data' / 'kelvins'
DATA_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DATA_DIR / 'kelvins_dataset.zip'
ZENODO_URL = 'https://zenodo.org/records/4463683/files/Collision%20Avoidance%20Challenge%20-%20Dataset.zip'

if not any(DATA_DIR.rglob('*.csv')):
    print(f'Downloading dataset from Zenodo (~220 MB)...')
    resp = requests.get(ZENODO_URL, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    downloaded = 0
    with open(ZIP_PATH, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f'\r  {downloaded/1e6:.1f} / {total/1e6:.1f} MB ({pct:.0f}%)', end='')
    print('\n  Download complete.')

    print('Extracting outer zip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)
    ZIP_PATH.unlink()
    print('  Done.')
else:
    print('Dataset already downloaded.')

# The training data is inside a nested zip — extract it if needed
inner_zips = list(DATA_DIR.rglob('train_data.zip'))
for iz in inner_zips:
    print(f'Extracting nested zip: {iz.name}...')
    with zipfile.ZipFile(iz, 'r') as zf:
        zf.extractall(iz.parent)
    iz.unlink()
    print('  Done.')

# Show all CSV files
for f in sorted(DATA_DIR.rglob('*.csv')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.relative_to(DATA_DIR)}  ({size_mb:.1f} MB)')

---
## 2. Load and Explore the Data

In [ ]:
# Find the training CSV (may be nested in a subfolder after extraction)
train_csvs = list(DATA_DIR.rglob('train*.csv'))
test_csvs = list(DATA_DIR.rglob('test*.csv'))

print(f'Found train files: {[str(f.relative_to(DATA_DIR)) for f in train_csvs]}')
print(f'Found test files:  {[str(f.relative_to(DATA_DIR)) for f in test_csvs]}')

# Load the first matching train file
df_raw = pd.read_csv(train_csvs[0])
print(f'\nTraining data: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
print(f'Unique events: {df_raw["event_id"].nunique()}')
print(f'\nColumns:\n{list(df_raw.columns)}')
df_raw.head(3)

---
## 3. Prepare Training Data

Each event has multiple CDMs at different `time_to_tca` values.
For the real-time pipeline we want: **given the latest CDM, predict the risk.**

We take the CDM closest to TCA for each event (the one with smallest `time_to_tca`)
as the label, and use the most recent available CDM as features.

In [ ]:
# For each event, take the CDM closest to TCA (smallest time_to_tca)
idx_latest = df_raw.groupby('event_id')['time_to_tca'].idxmin()
df = df_raw.loc[idx_latest].copy().reset_index(drop=True)

print(f'Events (one CDM per event): {len(df)}')
print(f'Target column "risk" — log10(collision probability):')
print(f'  min:  {df["risk"].min():.2f}')
print(f'  max:  {df["risk"].max():.2f}')
print(f'  mean: {df["risk"].mean():.2f}')

# Distribution of risk
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0a0a1a')
ax.set_facecolor('#0a0a1a')
ax.hist(df['risk'], bins=80, color='#00ff88', edgecolor='none', alpha=0.8)
ax.set_xlabel('risk (log₁₀ collision probability)', color='white')
ax.set_ylabel('Count', color='white')
ax.set_title('Distribution of Final Risk Score', color='white')
ax.tick_params(colors='white')
ax.grid(True, alpha=0.15, color='white')
ax.axvline(x=-6, color='#ff4444', linestyle='--', linewidth=1.5, label='10⁻⁶ threshold')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
plt.tight_layout()
plt.show()

high_risk = (df['risk'] >= -6).sum()
print(f'\nHigh-risk events (risk >= 10⁻⁶): {high_risk} ({high_risk/len(df):.1%})')

---
## 4. Feature Selection and Preprocessing

We select the most physically meaningful features and handle missing values.
The dataset has ~103 columns — many are correlated covariance terms.
We pick a focused subset that maps to what our pipeline can compute.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pickle

# Core features — the physical quantities that drive collision risk
feature_cols = [
    # Encounter geometry
    'miss_distance',           # relative position magnitude at TCA (m)
    'relative_speed',          # relative velocity magnitude at TCA (m/s)
    'relative_position_r',     # radial component (m)
    'relative_position_t',     # along-track component (m)
    'relative_position_n',     # cross-track component (m)

    # Position uncertainty (both objects)
    'c_sigma_r', 'c_sigma_t', 'c_sigma_n',  # chaser uncertainty (m)
    't_sigma_r', 't_sigma_t', 't_sigma_n',  # target uncertainty (m)

    # Object properties
    'c_rcs_estimate',          # chaser radar cross-section (m²)
    't_rcs_estimate',          # target radar cross-section (m²)
    'c_span',                  # chaser collision size (m)
    't_span',                  # target collision size (m)

    # Orbit properties
    'c_h_apo', 'c_h_per',     # chaser apogee/perigee altitude (km)
    't_h_apo', 't_h_per',     # target apogee/perigee altitude (km)

    # Orbit determination quality
    'c_obs_used',              # chaser observations used
    't_obs_used',              # target observations used
    'c_time_lastob_end',       # chaser time since last obs (days)
    't_time_lastob_end',       # target time since last obs (days)

    # Space weather (affects drag uncertainty)
    'F10', 'AP',

    # Time context
    'time_to_tca',             # how far from closest approach (days)
    'max_risk_estimate',       # max collision prob via covariance scaling
]

# Check which features actually exist in the data
available = [c for c in feature_cols if c in df.columns]
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    print(f'Features not found in data (will skip): {missing}')
feature_cols = available

print(f'Using {len(feature_cols)} features')

# Drop rows with missing target
df_model = df[feature_cols + ['risk']].dropna(subset=['risk']).copy()

# Fill missing feature values with median
for col in feature_cols:
    if df_model[col].isna().any():
        median_val = df_model[col].median()
        n_missing = df_model[col].isna().sum()
        df_model[col] = df_model[col].fillna(median_val)
        print(f'  Filled {n_missing} missing values in {col} with median={median_val:.2f}')

print(f'\nFinal dataset: {len(df_model)} events, {len(feature_cols)} features')
df_model[feature_cols].describe().round(2)

---
## 5. Train/Test Split and Model Training

In [ ]:
X = df_model[feature_cols]
y = df_model['risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set : {len(X_train)} events')
print(f'Test set     : {len(X_test)} events')

# Train gradient boosted regressor
# We predict log10(Pc) directly — it's already a continuous target
model = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    min_samples_leaf=10,
    random_state=42,
)

print('\nTraining...')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'\nTest set performance:')
print(f'  RMSE : {rmse:.3f} (in log₁₀ units)')
print(f'  MAE  : {mae:.3f}')
print(f'  R²   : {r2:.4f}')

---
## 6. Evaluation Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0a0a1a')

# --- Predicted vs Actual ---
ax = axes[0]
ax.set_facecolor('#0a0a1a')
ax.scatter(y_test, y_pred, alpha=0.3, s=5, color='#00ff88')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
        color='#ff4444', linestyle='--', linewidth=1.5)
ax.set_xlabel('Actual risk (log₁₀ Pc)', color='white')
ax.set_ylabel('Predicted risk (log₁₀ Pc)', color='white')
ax.set_title(f'Predicted vs Actual (R²={r2:.3f})', color='white')
ax.tick_params(colors='white')
ax.grid(True, alpha=0.15, color='white')

# --- Residual Distribution ---
ax = axes[1]
ax.set_facecolor('#0a0a1a')
residuals = y_test - y_pred
ax.hist(residuals, bins=60, color='#44ddff', edgecolor='none', alpha=0.8)
ax.set_xlabel('Residual (actual - predicted)', color='white')
ax.set_ylabel('Count', color='white')
ax.set_title(f'Residuals (MAE={mae:.2f})', color='white')
ax.tick_params(colors='white')
ax.grid(True, alpha=0.15, color='white')
ax.axvline(x=0, color='#ff4444', linestyle='--', linewidth=1)

# --- Residual vs Actual (check for bias) ---
ax = axes[2]
ax.set_facecolor('#0a0a1a')
ax.scatter(y_test, residuals, alpha=0.3, s=5, color='#ff8844')
ax.axhline(y=0, color='#ff4444', linestyle='--', linewidth=1)
ax.set_xlabel('Actual risk (log₁₀ Pc)', color='white')
ax.set_ylabel('Residual', color='white')
ax.set_title('Residuals vs Actual', color='white')
ax.tick_params(colors='white')
ax.grid(True, alpha=0.15, color='white')

plt.tight_layout()
plt.show()

---
## 7. Feature Importance

In [ ]:
importances = model.feature_importances_
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0a0a1a')
ax.set_facecolor('#0a0a1a')

colors_cycle = ['#00ff88', '#44ddff', '#ff8844', '#ff4444', '#aa66ff']
bar_colors = [colors_cycle[i % len(colors_cycle)] for i in range(len(feature_cols))]
ax.barh(range(len(feature_cols)), importances[sorted_idx], color=bar_colors)
ax.set_yticks(range(len(feature_cols)))
ax.set_yticklabels(np.array(feature_cols)[sorted_idx], color='white', fontsize=9)
ax.set_xlabel('Feature Importance', color='white')
ax.set_title('Gradient Boosting — Feature Importance', color='white')
ax.tick_params(colors='white')
ax.grid(True, alpha=0.15, color='white', axis='x')

plt.tight_layout()
plt.show()

print('\nTop 10 features:')
for i in sorted_idx[::-1][:10]:
    print(f'  {feature_cols[i]:<30} {importances[i]:.4f}')

---
## 8. Export Model for Pipeline Use

Save the trained model so `src/detector.py` can load and score conjunction events.

In [ ]:
model_path = Path('..') / 'data' / 'collision_model.pkl'

with open(model_path, 'wb') as f:
    pickle.dump({
        'model': model,
        'feature_cols': feature_cols,
        'version': '0.1.0',
        'dataset': 'ESA Kelvins Collision Avoidance Challenge',
        'training_samples': len(X_train),
        'rmse': rmse,
        'r2': r2,
    }, f)

print(f'Model saved to {model_path}')
print(f'  Features : {len(feature_cols)}')
print(f'  RMSE     : {rmse:.3f}')
print(f'  R²       : {r2:.4f}')
print(f'  Size     : {os.path.getsize(model_path) / 1024:.1f} KB')

In [ ]:
def score_conjunction(features_dict, model_path='../data/collision_model.pkl'):
    """
    Score a conjunction event from a dict of CDM features.
    Returns log10(collision probability).

    Usage:
        risk = score_conjunction({
            'miss_distance': 150.0,      # meters
            'relative_speed': 14200.0,   # m/s
            'c_sigma_r': 50.0,           # meters
            ...
        })
        probability = 10 ** risk
    """
    with open(model_path, 'rb') as f:
        bundle = pickle.load(f)

    features = pd.DataFrame([features_dict])

    # Fill any missing features with 0 (will use median in production)
    for col in bundle['feature_cols']:
        if col not in features.columns:
            features[col] = 0

    return bundle['model'].predict(features[bundle['feature_cols']])[0]

# Quick test with a high-risk scenario
test_risk = score_conjunction({
    'miss_distance': 50.0,
    'relative_speed': 14000.0,
    'c_sigma_r': 20.0, 'c_sigma_t': 500.0, 'c_sigma_n': 15.0,
    't_sigma_r': 10.0, 't_sigma_t': 200.0, 't_sigma_n': 8.0,
    'c_rcs_estimate': 5.0, 't_rcs_estimate': 15.0,
    'c_span': 2.0, 't_span': 4.0,
    'c_h_apo': 420, 'c_h_per': 415,
    't_h_apo': 420, 't_h_per': 418,
    'time_to_tca': 0.1,
})
print(f'Test risk score: {test_risk:.2f} (log₁₀ Pc)')
print(f'Collision probability: {10**test_risk:.2e}')

---
## 9. What We Have & What's Next

### What this notebook established:
- Trained on **real CDM data** from ESA's Collision Avoidance Challenge (~13k events)
- Gradient boosted regressor predicts log₁₀(collision probability) from CDM features
- Exported model to `data/collision_model.pkl`
- `score_conjunction()` — ready for `src/detector.py`

### What's next — `src/` pipeline services:
```
✅ 01_explore_tle_data.ipynb         
✅ 02_fast_conjunction_search.ipynb
✅ 03_ml_model_training.ipynb
⬜ src/ingester.py                   (Kafka producer — fetch TLEs, publish to Kafka)
⬜ src/propagator.py                 (Kafka consumer — propagate orbits, write to Redis)
⬜ src/detector.py                   (conjunction search + ML scoring)
⬜ dashboard/                        (CesiumJS frontend)
```